In [ ]:
# import
from inference import *
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
import numpy.random as npr

def lo_histogram(x, bins):
    """
    Left-open version of np.histogram with left-open bins covering the interval (left_edge, right_edge]
    (np.histogram does the opposite and treats bins as right-open.)
    Input & output behaviour is exactly the same as np.histogram
    """
    out = np.histogram(-x, -bins[::-1])
    return out[0][::-1], out[1:]

def gamma_isi_point_process(rate, shape):
    """
    Simulates (1 trial of) a sub-poisson point process (with underdispersed inter-spike intervals relative to Poisson)
    :param rate: time-series giving the mean spike count (firing rate * dt) in different time bins (= time steps)
    :param shape: shape parameter of the gamma distribution of ISI's
    :return: vector of spike counts with same shape as "rate".
    """
    sum_r_t = np.hstack((0, np.cumsum(rate)))
    gs = np.zeros(2)
    while gs[-1] < sum_r_t[-1]:
        gs = np.cumsum( npr.gamma(shape, 1 / shape, size=(2 + int(2 * sum_r_t[-1]),)) )
    y, _ = lo_histogram(gs, sum_r_t)

    return y

class HMM_Step():

    def __init__(self, m=50, r=10, x0 = 0.2, Rh=50, T = 100, isi_gamma_shape = None):
        
        self.m = m
        self.r = r
        self.x0 = x0
        self.p = r / (m+r)
        self.Rh = Rh
        self.T = T
        self.dt = 1/T
        self.isi_gamma_shape = isi_gamma_shape

        self.states = np.arange(self.r+1)

        self.transition_matrix = np.zeros([self.r+1,self.r+1])
        for i in range(self.r):
            self.transition_matrix[i][i] = 1 - self.p
            self.transition_matrix[i][i+1] = self.p
        self.transition_matrix[self.r][self.r] = 1

        self.initial_distribution = np.zeros(self.r+1)
        self.initial_distribution[0] = 1
        for i in range(self.r):
            self.initial_distribution = np.matmul(self.initial_distribution, self.transition_matrix)

        self.lambdas = np.ones(r+1) * self.x0 * self.Rh * self.dt
        self.lambdas[-1] = self.Rh * self.dt

    def emit(self, rate):

        if self.isi_gamma_shape is None:
            # poisson spike emissions
            y = npr.poisson(rate * self.dt)
        else:
            # sub-poisson/underdispersed spike emissions
            y = gamma_isi_point_process(rate * self.dt, self.isi_gamma_shape)

        return y

    def simulate(self):
        latent = np.empty(self.T)
        rate = np.empty(self.T)
        latent[0] = np.random.choice(self.states, p=self.initial_distribution)
        for i in range(1, self.T):
            latent[i] = np.random.choice(self.states, p=self.transition_matrix[int(latent[i-1])])
        for i in range(self.T):
            if latent[i] == self.r:
                rate[i] = self.Rh
            else:
                rate[i] = self.Rh * self.x0
        spikes = self.emit(rate)
        return latent, rate, spikes
    
class HMM_Ramp():

    def __init__(self, bet=0.5, sig=0.2, K=100, x0 = 0.2, Rh=50, T = 100, isi_gamma_shape = None):
        
        self.bet = bet
        self.sig = sig
        self.K = K
        self.x0 = x0
        self.Rh = Rh
        self.T = T
        self.dt = 1/T
        self.isi_gamma_shape = isi_gamma_shape

        self.states = np.arange(self.K)

        s = np.linspace(0,1,num = K)
        self.transition_matrix = np.empty([K,K])
        for i in range(K-1):
            arr = (s - s[i] - bet*self.dt) / (sig*np.sqrt(self.dt))
            dist = normal_dist(arr)
            dist_norm = dist / np.sum(dist)
            self.transition_matrix[i] = dist_norm
        self.transition_matrix[K-1] = np.zeros(K)
        self.transition_matrix[K-1][K-1] = 1

        arr = (s - x0) / (sig*np.sqrt(self.dt))
        dist = normal_dist(arr)
        self.initial_distribution = dist / np.sum(dist) 

        self.lambdas = np.arange(self.K)/(self.K-1)*self.Rh*self.dt

    def emit(self, rate):

        if self.isi_gamma_shape is None:
            # poisson spike emissions
            y = npr.poisson(rate * self.dt)
        else:
            # sub-poisson/underdispersed spike emissions
            y = gamma_isi_point_process(rate * self.dt, self.isi_gamma_shape)

        return y

    def simulate(self):
        latent = np.empty(self.T)
        rate = np.empty(self.T)
        latent[0] = np.random.choice(self.states, p=self.initial_distribution)
        for i in range(1, self.T):
            latent[i] = np.random.choice(self.states, p=self.transition_matrix[int(latent[i-1])])
        for i in range(self.T):
            rate[i] = latent[i]/(self.K-1)*self.Rh
        spikes = self.emit(rate)
        return latent, rate, spikes

def normal_dist(x, mean = 0, sd = 1):
    prob_density = (np.pi*sd) * np.exp(-0.5*((x-mean)/sd)**2)
    return prob_density

In [ ]:

#common parameters
x0 = 0.2
Rh = 75
T = 100

# define parameters step
m = 32
r = 20

shmm = HMM_Step(m, r, x0, Rh, T)
latent_step, rate_step, spike_step = shmm.simulate()
ll_step = poisson_logpdf(spike_step, shmm.lambdas)
posterior_step, normalizer_step = hmm_expected_states(shmm.initial_distribution, shmm.transition_matrix, ll_step)

prob = np.array([i[-1] for i in posterior_step])
actual = np.array([1 if i == Rh else 0 for i in rate_step])

plt.plot(np.arange(T), prob, label = 'Computed Firing Probability')
plt.plot(np.arange(T), actual, label = 'Actual Firing State')
plt.xlabel('Time index ($ \Delta t = {:d} ms$)'.format(int(1000/T)))
plt.ylabel('$s_t$')
plt.legend(loc='lower right')
plt.title('Smoothing of Step model: $m$={:d}, $r$={:d}'.format(m,r))
plt.show()

In [ ]:
# import
from inference import *
from HMM_models import *
import matplotlib.pyplot as plt

#common parameters
x0 = 0

T = 500
dt = 0.01  # time step
N = 10  # number of trials

# define parameters ramp
beta = 2
sigma = 0.5
K = 50
Rh = 30

# define parameters step
m = 250
r = 10
R_low = 5.0  # low state firing rate
R_high = 50.0  # high state firing rate

# Replace numerical arguments with variable name
rhmm = HMM_Ramp(beta, sigma, K, x0, Rh, T)
shmm = HMM_Step(m, r, x0, Rh, T)

latent_ramp, rate_ramp, spike_ramp = rhmm.simulate()
latent_step, rate_step, spike_step = shmm.simulate()

ll_ramp = poisson_logpdf(spike_ramp, rhmm.lambdas)
ll_step = poisson_logpdf(spike_step, shmm.lambdas)

posterior_ramp, normalizer_ramp = hmm_expected_states(rhmm.initial_distribution, rhmm.transition_matrix, ll_ramp)
posterior_step, normalizer_step = hmm_expected_states(shmm.initial_distribution, shmm.transition_matrix, ll_step)

expected_ramp = np.array([np.matmul(rhmm.states, i) for i in posterior_ramp])
expected_step = np.array([np.matmul(shmm.states, i) for i in posterior_step])

plt.plot(np.arange(T), latent_step, label = 'Actual states')
plt.plot(np.arange(T), expected_step, label = 'Expected states')
plt.imshow(posterior_step.T, cmap='viridis', interpolation='bilinear', origin='lower', vmin=0.0, vmax=0.3, aspect = 'auto')
plt.xlabel('Time index ($ \Delta t = {:d} ms$)'.format(int(1000/T)))
plt.ylabel('$s_t$')
plt.colorbar()
plt.legend(loc='lower right')
plt.title('Smooth Step model: $m$={:d}, $r$={:d}'.format(m,r))
plt.show()

plt.plot(np.arange(T), latent_ramp, label = 'Actual states')
plt.plot(np.arange(T), expected_ramp, label = 'Expected states')
plt.imshow(posterior_ramp.T, cmap='viridis', interpolation='bilinear', origin='lower', vmin=0.0, vmax=0.3, aspect = 'auto')
plt.xlabel('Time /ms')
plt.ylabel('$s_t$')
plt.colorbar()
plt.legend(loc='lower right')
plt.title('Smooth Ramp model: $\\beta$={:.1f}, $\sigma$={:.1f}'.format(beta,sigma))
plt.show()

In [ ]:

posterior_ramp, normalizer_ramp = hmm_expected_states(rhmm.initial_distribution, rhmm.transition_matrix, ll_ramp, filter=True)
posterior_step, normalizer_step = hmm_expected_states(shmm.initial_distribution, shmm.transition_matrix, ll_step, filter=True)

expected_ramp = np.array([np.matmul(rhmm.states, i) for i in posterior_ramp])
expected_step = np.array([np.matmul(shmm.states, i) for i in posterior_step])

plt.plot(np.arange(T), latent_step, label = 'Actual states')
plt.plot(np.arange(T), expected_step, label = 'Expected states')
plt.imshow(posterior_step.T, cmap='viridis', interpolation='bilinear', origin='lower', vmin=0.0, vmax=0.3, aspect = 'auto')
plt.xlabel('Time /ms')
plt.ylabel('state, $s_t$')
plt.colorbar()
plt.legend(loc='lower right')
plt.title('Filtered Step model: $m$={:d}, $r$={:d}'.format(m,r))
plt.show()

plt.plot(np.arange(T), latent_ramp, label = 'Actual states')
plt.plot(np.arange(T), expected_ramp, label = 'Expected states')
plt.imshow(posterior_ramp.T, cmap='viridis', interpolation='bilinear', origin='lower', vmin=0.0, vmax=0.3, aspect = 'auto')
plt.xlabel('Time /ms')
plt.ylabel('state, $s_t$')
plt.colorbar()
plt.legend(loc='lower right')
plt.title('Filtered of Ramp model: $\\beta$={:.1f}, $\sigma$={:.1f}'.format(beta,sigma))
plt.show()

In [ ]:
# Explore beta vs sigma
# Change these two linspaces to increase resolution and range of heatmap
# I suggest the two ranges below for best representation, but takes about 3 mins to run
# beta_vals = np.linspace(0.05, 1, 30)
# sigma_vals = np.linspace(0.005, 1, 30)
# ie use a resolution of 30
# Having a resolution of 6 shows more granular plot but should run in about 10 seconds
import inference_analysis as IA

ramp_resolution = 6
beta_vals = np.linspace(0.05, 1, ramp_resolution)
sigma_vals = np.linspace(0.005, 1, ramp_resolution)

errors = np.zeros((len(beta_vals), len(sigma_vals)))

for i, beta in enumerate(beta_vals):
    for j, sigma in enumerate(sigma_vals):
        _, _, mae = perform_ramp_inference(
            K=K, beta=beta, sigma=sigma, dt=dt, T=T, R_h=R_h, N=N
        )
        errors[i, j] = np.mean(mae)

IA.plot_error_heatmap(errors, beta_vals, sigma_vals,
                   'beta', 'sigma',
                   'Ramp Model Inference Error vs Parameters')